# Tutorial 6 · Convolution first, then convolutional neural networks
**CSCI3230 / ESTR3108 · 45-minute guided tutorial**

**Part A: understand the operation.** Choose filters by hand, inspect their responses, and
calculate shapes and parameter counts. Apply the same idea to signals and volumes.

**Part B: build a model.** Let the filters be learned, add nonlinearities and pooling, then
connect the features to a classifier. Inspect a model trained before class.

By the end, you should be able to:
1. Calculate a convolution output and explain local connections and weight sharing.
2. Explain why stride and padding are chosen, and predict output shapes and parameter counts.
3. Distinguish a convolution operation from a complete CNN.
4. Trace activations through a CNN and explain how convolution extends beyond images.

**How to participate:** predict first, run the next cell, then explain the result.
All supplied code is complete. Change the marked settings for short experiments.

| Time | Topic | Activity |
|---|---|---|
| 0–3 min | Images and local patterns | Identify the dimensions |
| 3–10 min | One convolution and three filters | Hand calculation; predict A, B, C |
| 10–18 min | Stride and padding | Compare animated windows; calculate shapes |
| 18–23 min | Channels and parameters | Unpack RGB convolution; count shared weights |
| 23–28 min | 1D signals and 3D volumes | Compare signal height with local change |
| 28–34 min | From convolution to CNN: ReLU and pooling | Read a function graph; follow max pooling |
| 34–37 min | Receptive field | Trace one output back to its input region |
| 37–43 min | A complete CNN | Trace shapes; inspect pretrained features and predictions |
| 43–45 min | Exit questions | Explain the main ideas |

The short 1D CNN at the end of Part B can be skimmed if time is tight.
The optional architecture comparison and **after-class training** are outside the core 45 minutes.
No training starts by default, including with **Run all**.

## Setup · Run this first

Open this notebook in **Google Colab** (upload the `.ipynb`) or local Jupyter.
Use a CPU runtime. The classroom examples use PyTorch, NumPy and Matplotlib.
For local use, install `torch numpy matplotlib jupyter`; `torchvision` is needed only for training.

Thirty small MNIST test images are embedded below, so the basic lesson needs no dataset download.
For the pretrained demonstration, put **`tutorial6_cnn.pt`** in the notebook's working directory.
The TA will provide this file separately after training.

**Without a checkpoint:** all basic calculations, shapes and the 1D example still run.
CNN feature plots are explicitly labelled **untrained**; trained predictions and training curves
are skipped. No training runs by default, including when you choose **Run all**.

In [ ]:
from pathlib import Path
import base64
import zlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)
torch.set_num_threads(2)
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=["#751475", "#2878a5", "#bf7b19", "#20846d"])
})
PURPLE = "#751475"
CHECKPOINT_PATH = Path("tutorial6_cnn.pt")
print("PyTorch:", torch.__version__, "| Classroom device: CPU")
print("Working directory:", Path.cwd())

In [ ]:
# Embedded MNIST examples: first three test examples of each class, ordered by round then class.
# uint8 pixels are converted to float32 in [0, 1]; no extra normalization is used.
_mnist_bytes = zlib.decompress(base64.b64decode('eJztnHt8Ddfax6fSIARBosc9dUmRqkuKRutSl9AqSom2qL6lwdFSSTglLWlDEOQoRyK0qFskqki8jSgS1USIiIi4R0giN5HmJpHseZ45a2bvneyZWU/+eD/nnPcUPx97L/trzazLzJq1nudZIwj/f2ocBOc7EqyrwQDz+MghjmTzTxkYC/t8MIeBQWYGw20XHfpFAqaCO+xDi4aky/k2jR3sazDMVSPHXHa422saCULHHEPZQmtt8U/YK8nP2f/qrGHnOhiTHRO0rLYEjucl2GPJ1hkM5qTumDfMzGEIK1VOBy7bwGqSPkjgsV/SGTuqrt9NgLfeygZQGkeNhIVKUxo/NmkYaw0jyznZuZGGCYMDjYzff6N/NhwcNboDlz0dcsLPSTZFnEiyVSUk6lkeRLJJOIRk5zMaU8gRr5PZZuAZkq3DsRRyfZjUkGJfSXvJQx7ACRT6S941MtuXuINkwRhIsmwcRqFBBpqtxyQrAjW6hkuobNbxh3V30JMk768PSFLQdB4KA0U3OYOBjNICDwMs1aFXq+Gyo61Q/yKs07Gxhsut2dfSx/Cm/pgdW8ifKcBjihZVQjzRF+9UQi51G/kCbCDQ4QrYYctHrQsgvzMfCfEA6wk07jGcJI7YMoHO5g9wkMgmPAZoTSCZOdszWQvW9vZdg4KCNjWyZIr2/32fKeVTw36GWlVVVoYtWuRae9DFPj4+cpZtPj7dqRM/dbLtvnFjbwJ9wxqrencLHltpbOP7bhz2KYjfecx7CJW++qv6fyGUfQ56ALDbWsskcJa/Bp4G2PO8hv0Kjsr3gEIAdw3byNisU1OmTNlkeSkY9RGM62a+YDImNVOxptNsOxfXXC7lH2hLO+bHyBqa4qwtrZWd3UtOdnbflQFEqxvQfqA5NZDd9VGWaGx65XhzusktKB5twT6sBLHmau1ngBjLjN4Ar5vTM0X4TlWPE5C+3UlOLUitADUTbNMrwFDFJNcioaWmEjOiso31O/PVC9oKsqdc14C3l73ftYGePL3qe9eUcGuvY0tyTYl/7Nei58+Y2ceXtXOfkaK/KeUpOqhRz8Ib5uEzVsv2V/YzpVpIoGaTSlPNyfVwUn2jhYl/NaUc86rVs4pm90Rz0l9MVSGhlVizYggTQ9XM5kKKaZBoBbrpd5h4djLTN3vOiDVnNqt7+CORKS/XIIo2glZ9JjEJwo+iqENmLRfFnhTz1S+5LPORbJVYTrK8wgUkiySnaE+Imvfq1cvum8m99GRMyA02hlyr0LV458BysWYM1bAhpp/TwpksfrdfMVpwLcoq2v/VSHvNrdL4Ioxj163QoZ6uCPWPwApiEmW7EvKb8ZEwDTLaEUgIgp8pJBRAxfI+BJNXtYZNU5dM7tFjsvbgay0mKHmaUcKq3807BjMUv9Idd/joBBM8xDnrl1C1xWUPn/VleU6wjvgHh9mEykesPsRdkb1wNBfSfXlE1vTNrSj0p1eDC3CYQkGi+DXBFonHX6MOuV5cTJ5ua2VfCrWB38lsQTXMdfJkJzW7Bx8r38GZZaJYpCpwo/uZ7PP5/vdBKjh0T8q2NJXNF9mavo2vKGb5s4s3QrSszg5xoXxO8bizsT6WLJoxp4cQXF/51/rz9S1YPHgKm2C3ua5xlmWJY/l+Ek+Y63pAy1rmVi5Vpi0J5Zana3NHLkvrRPF0E+GdhCp1f0RDpDwWvNO5/VYo13RVuxT4faKrq1+cCFd16/vW19njCNjf77UzJSa72WuL161d201PnulfqknRO4Lf6MJnd5CpJN6k8Fct2XDP0Z67s/Ae+x/V9xH16/Tmw5oOHz78dYdC1M5EavQepHAXPEyt8qX3qGzfYNHLBHq9CnlGY0Ur8VfdMsgkm6THAwkkLMNfKDTGUOxKoJbpuI9AVol4i7IKOCFpWut4F72eI9hKxFcJNKiUZksQb1F32BJMpvrmzyTnXpwplCKbz9gy7VJycvLmN5to0TFkj2nj3zS13WRAIuLZ5cvfmD5pRPAd/FHFAgDOmoeybZCqZc1NybdLNGwugIcx1ek+wE51WTbjfbmLnELYjXZUY8NpGY/JNp1GswX+3bXaOgjNU+FaMUKBb3MtEQQPdh7IHqvLI7iFo6K/aUGbb3NYe0R7DU2Fx5q10b48gMpdrzRpMPURoOaBzNYFcHn68h0X5PNp54X+JSgZT5fdSVeStl7rLl68mHzRh5pNPtO/Vk03n6aGx6l3ATjPaFntHrDLYh9/pNig3EJFXvX1qGMJXIpmMPcvejYeTwsNP7mF0jn9Yd3hXfb5C2CM3tJ5CLYJ8mIBV+mP6Q6Xuk3eZ3iIhT10rEWRXM7oLtdhiz7jiGIJv2so+GMGZ8gesT2QFcPmEPyoZya9D5nkMFpvHywjM/Z+BE4k9MKf9Et4kxxu4itkxg5Iu2CE4+X6xjGracY4OuN/tQZZ9Vh8evHiQXrSNPJRYak8/Dx6MEnLgtnPabEREUfZBEjTnM4PMHNoO9bx9XxF6aB6dH0Na2wy/tU4RsWG4PaadDr+oGK/4eyadBCmWaJOt/+onZtMUrNlaDGF17Crf7xBs7MCxRrfotknaMl2YQrFXIpUdbdkLnvxjKXBeVhJDbMKxawBlsUUrqYZne+vbDmPWhfnVbxwVNYDxILvNdaICUnG5wqKBV8KWrW5rKCQOTryTLLG4jzKydkyC5EaeCYg7iXmpw3YpOktIls/tn6gSuLPpjwU+x0fE64UYSDiQyrbAsQVFNuNRZR96g0RM6hs4xG31nHIfgRqJ+JlKps7oh/F5mKBPcUO4QXqGW6diqRpw2o77qSY0OYHIoDnSVGXr2NusnloiT6gSLhQZTICVj/QkPdLDHAvsI+trS2bvavt1DPuQejCpkrSLhUWWaKpRRBec4kFqdgnhpu9a/tOzb6AL2r/YXOs1HJGYd/ToseHQzG3ikxtCx7PJdDLPwB1TzQ+CYltCfQ9VBBrVYbOjyBR5ij9zx1efHHd7uOQqZ/y2E4oN/XBcbcXBKHV97XnfPeM8vvjgn2PAYqD3towp6YLGxj9kKWsyvNKWeLRztojulazHypXKasYh0VRUaphZInP4VV2/KI/EymHhaeRTaLxJ05kUxSIxj9lelNDopQf5ZGYyObswTo2IVi20nY/LQLhBmqchpDIRz5p7Hz8gCIXuZzBDlzW6AorZ17GUn58lk+GUkfO1ctk39ePnTOWXxzWQLvQvMbXs0QQObNTRSsApxK5AkHM54fYdb8K6ulerXaXgfgTx3XT+OBBkOAut8lWyL13gP/MCYHyRCqabVqsPrDniVKrWH9Hc7rZWNWDt/mD6rAadLukqwWyPwm1oZZrYZZlNjeodS4740FL+0yrEJOLQUa5OM0y227pQo1HaY7FKkPWLogwlczG76HGLbgLIOaQG9PKOIAwNXPJNlrA5A+dubL5m2swb82aNS8DvfTthBf5dxnTThhJoclYQnpnttexes0tJ7PNwTwy2yX4QWhCRKpegq1TLxD1u8TaZas+KkHRoFO+L3BsHk+KXFaw8QUSd2njujwCExPlcT6YE6eEbJwI9OCPFD+JxCjO5JBRQIcRLwWyywUfnOri4sJ1UjtkKM8j3rPK4Sqrl4dHIhvpdXkHmdxl9l/EwhU6ps0jsYAzkTTJPi2Pjv0dnFjwBQnt0+ggCjYKk6h7/hXLfy60GDc6Hiy3fCJPBNMz2MHDIyr/iupRMFGCgi0hpw/Kz7+0QM2jYFRwcB5ciQ3269v3iY1y7XMUyPHlmCjm7eGjYQXiaie+j7ploXhYG3hn1juiSDq218JBCrW6IpLDqjcUkzfLD3CRQkIOOW1hQ650NOAomzbc+R+9vSBHduyKqezvIg4rjvpgZIORB0RxspatghDlu8k10VvLvM3l3KZnf5VSldC5PvmgY8IOcasNW2PEieIMHWueJW51m5AtZn3LqWHfLNlDHc1vud6R4qlFT/A4/9+gt6tk82xo6OrW776hZXNMNj6sQjFu4aiXLJn1zFVz53rGxsbmK/+j4hvewXt6eiYyWEx4mBp23obIuyoUdcESvdPKJC9EKvqkUxkWcxxyihYgriVQ1zIsJzy19ocRFxLZ1iHe1odvKXofsIwI5xtSgkg43+3KEDfxTUz1WPETCevTQNbMOhO7UU0fovQbMRBORPyNsD4IL+WcJoPJ/lyqP0b4JDriI72DrdmCM0UfCZfYLaAZ0uz7zouriGrN1siPsEq1PmwxKQcx3V0QnvsaMEJtelyJeGMxa+nB57HET71JZ145pskXw+BivK2Z1L5cgSly8F3HEkzX9sUMRNle3zweb+r6ou0tzN/wUo8LmMSZFHg9RizPxwyO908QnCbtf8Sul2QvrqW5aSHm7H5Ynvoux9Y8E3Emu1fu4bf6PUij8L7sMqwXjvrp2ceIikltAsbr2Lsm5qf3iQvWV1H2ij0XVc7ZNzIUWPXqLcMIXi2mFN352xG8wb+4W2z9/fdPye0az8RTl9fWnDs7oyuH9NxSoDwDqlODNQ/rV0KKEbP2rTQk4P17am9QCMvzayC7imK6xVdkWEZ8N1wGmO+rLLcvO7ux41qw0WWY3V9OWDl63ytFaZfFcDimGDM+D4iICLtsyGW58iwtnTaHykFCNBhdVgc0XW+3+szhjSEJMgvimyZ3IZbM5PtsFlcjfsglwqxSxFT+foP+7AlQ+joXsRsBy4fyURM2VHF86LJssxEvETsDx7G6U/HcKYhrCCRkSfnkXbJQ+oxCfyY1HxYgQVgAZ7+I9ZfZxujJ7Xr2mTz3PiV/6JBzPvvV23oVhzlHgJjuXE+w7p8naj1T00Gq8lJSqw2Sxtu1T4RoUzJFs5JoeVsE84rTW8PcRDG2eQ3LUPXkUlGsGVG9Qb2qWWER3O4tqQ1Jxyxqpc2nZurVkCW7DOpVeO+82ibOydNcwtNrloXbQLsPpdMeMM7LhhXm6gIrHO8U+7NH3dQCkXN5tz5SnDybddRhXphgs/6R4po1/am14xMgXymGZDGSNJRikiT51sHIY9bFfOs+5n+aUejfwmLqKEsdbGgddYihmZxtKHlIsiBPgfqGI/Es7hLDZsq8TSiClVspPsjnhi6/cAzLZradz2P2SZWftBGE+bhZz7rNUex+8/FFsqDzC0jkeJVzSJP8E0iPq30ute1aEDaWj6aQ3TkyyEPYk2negdF4aZI6OHi8+TJrNul69QZVhGGD5Gxlcmm/ugizNbP9Vcrspd8pxDitscsh9xfB8dNjVX9EfqwbsH1xpm8eGqI5fft8ujzbi+Vt+BUGIJ6YO5p8p8PTq5lym91YzDPNR1QaQwj0uxSEoeVw6ciRATOhapqOtdwbKU9LRgKQb/MYCVV6m6JRTULg7wRyLQb4gEvqL2BFzeK6Fzoatw+nv89hnc072S7qg6LaJSUlTRwx4gRAJuWZGBgKcJfYBCJYHQYgd0cIByGdt+pSNAdgDIG6ZxHMdsi6TIAi/b6K7j1CTstVvKHdkm7VfrtxJ1+ehy6g3c77OANZ3pSB7OnQfiKGgKleAR2C7IJ65hTR0cT0sQIfmayGU7C/ju007Qw4WaQbB5tk7TfaKH8r1GUbgJ7Kt92DUB3zQuPUezbqApsapBeO+f4oWwGBKb+Fmslr7NTIyMgKrPxIwxre+GORss0yE0vjtBmbGsOu2pYlv0Jd8TPp4ELBG4fXwagAkLrZz9Iskv1K2XCZvJ7s99HUrZFbHiJKKOp2RU3Z8xDwZsB7L23VbiEIqAC8HtpfNufMitEYWvIxrJ/JsNRZa+D4GbVXWI2cHqXrt9+YFMibf5kUjLSTOLPUkUK2xYkvM3EtMG5GN0rSZA7teixs5cpdVxD3Ua91snE5gtQOXaFtClJh1kNS8DdeC9QXGq0px3jOnrUmU5ZuysDK5fpcVourLz9G8Sxvl3iAbPs7x58qjg+Nm/Lnf+OGp0TtQmSXBM3aS7qx0eKQ2qdsreIlIu5FPmQmmW29ZHpxkau7+3pNocJNp3PNZItATYnNLFySPF0z43nMVZK/wyUeM36Fq+uzXql6eyle0LP2yoo43Ng48Zr6xMs/GzO4a5vPXZLCWUlY47hnZmqbyFOpmbt7uLZ6yinXS4ri+Q3Lsq6nuoPlDA+nunh9Zrgn2VfP9G+X7BpoPE2JmgHttuRAwcEjTcwP9vMDUfOeut1btlzNj1LiLlaApDJUTiwD0c8UEdXoCqg2CfgketRG5iRK+ohas3YBzTw4kbg1rI58u6k4VbksdcRpJdYV+0XHj46uIywlX6QZ5pOoex2180GyBg4ZRBSuIMcaR1FIcBHJXRz/efVdnhLhz3W39lOWzhjKiXMZ+ACvzxvnhaDfGej8EPwayXt79KxhFMhBDh1vYIJuY2TDizhQvgSR97rAQzCs89clkODKcXQdgoy7rJinN3IcqgGs+I+icwN485dRj9idGcv3wnbfg1IRf+/mZyWQfA4yed5rjxI407t7CWzUkQZbsNrbRr50q3XLo25gfHmkPWelHYc3lQAVZyzXxmsMrwZ5m0XD2Xmge4PNeIDxLcbvT0LYr+u7LlmARay9Yj/ndGuvHIQ7s+kF/zP9H9RHjpML69L4Hd1NZj3C6MfbeRJ0BqgBopg1fuTIWR8WVmrXjs75xrC0NomibrvzfohQFqhuFl40k7ZBqeKKso6BU9psyWKOglaL4lA+cwxgBdLZaveKVcnJydlyHbRIsIlU4gDHhHEDGod6e78tbIZrZJw4iLso5FhnQCMRj8narVjfzGZNB5EIUpLfFpdCIdY4+pBDs/bWwaYn0G/t1eifsbINtg=='))
demo_images = torch.from_numpy(np.frombuffer(_mnist_bytes, dtype=np.uint8).copy()).reshape(30, 1, 28, 28).float() / 255.0
demo_labels = torch.tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=torch.long)
demo_test_indices = [3, 2, 1, 18, 4, 8, 11, 0, 61, 7, 10, 5, 35, 30, 6, 15, 21, 17, 84, 9, 13, 14, 38, 32, 19, 23, 22, 26, 110, 12]

def matrix_plot(ax, values, title, cmap="coolwarm", limit=None):
    values = np.asarray(values)
    if limit is None:
        limit = max(float(np.abs(values).max()), 1.0)
    ax.imshow(values, cmap=cmap, vmin=-limit, vmax=limit)
    for (r, c), value in np.ndenumerate(values):
        ax.text(c, r, f"{value:g}", ha="center", va="center", fontsize=11,
                color="white" if abs(value) > limit * 0.65 else "black")
    ax.set(xticks=np.arange(values.shape[1]), yticks=np.arange(values.shape[0]), title=title)

def parameter_count(module):
    return sum(p.numel() for p in module.parameters())

def image_grid(images, titles, columns=5):
    rows = (len(images) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 2, rows * 2.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, img, title in zip(axes.flat, images, titles):
        ax.imshow(img.detach().cpu().squeeze(), cmap="gray", vmin=0, vmax=1)
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    plt.show()

print("Embedded image batch:", tuple(demo_images.shape))

### Optional: upload the checkpoint in Colab
Only needed if the TA's file is not already in the working directory.
Set the switch to `True` and run this cell once, choose `tutorial6_cnn.pt`, then reset it to `False`.
Local Jupyter users can simply copy the file next to the notebook.

In [ ]:
UPLOAD_CHECKPOINT = False
if UPLOAD_CHECKPOINT:
    from google.colab import files
    files.upload()
else:
    print("Upload skipped. Expected checkpoint:", CHECKPOINT_PATH)

# Part A · Convolution as an operation

First inspect filters chosen by hand. No training is needed for this part.

## 1 · Images and local patterns

Before building a network, we will study how a local filter processes an image.
Last week's MLP gives us a useful comparison for how the input is arranged.

A grayscale digit has shape `[1, 28, 28]`. A batch has shape `[B, 1, 28, 28]`.
The dimensions mean **batch, channels, height, width**.

An MLP can flatten each image into 784 values. Flattening is reversible: the pixel values
are not destroyed. However, a fully connected layer does not explicitly impose a preference
for nearby pixels or reuse a local detector across positions.

A convolutional layer introduces:
- **Local connections:** each output initially uses a small patch.
- **Weight sharing:** the same filter is reused at every spatial position.

**Predict:** if a batch contains 10 images, what is the shape after flattening each image?

In [ ]:
image_grid(demo_images[:10], [f"Label: {int(y)}" for y in demo_labels[:10]])
print("Image batch:", tuple(demo_images[:10].shape))
print("Flatten per image:", tuple(demo_images[:10].flatten(start_dim=1).shape))

## 2 · One convolution = multiply, sum, add bias

For a single input channel, a `3×3` filter uses nine values at each position:

$$y_{r,c}=\sum_{u=0}^{2}\sum_{v=0}^{2}K_{u,v}X_{r+u,c+v}+b.$$

For this first example, stride is 1, padding is 0, and bias is 0.
The kernel below responds to a **left-to-right intensity change** (a vertical boundary).
Positive and negative responses indicate opposite directions of change.

**Pause before running:** take the top-left `3×3` patch. What is the sum of the nine products?

*Terminology:* PyTorch's “convolution” uses cross-correlation: it does not flip the kernel.
We use that convention throughout, matching the usual CNN implementation.

In [ ]:
small_image = torch.tensor([
    [0., 0., 1., 1., 1.],
    [0., 0., 1., 1., 1.],
    [0., 0., 1., 1., 1.],
    [0., 0., 1., 1., 1.],
    [0., 0., 1., 1., 1.],
])
edge_kernel = torch.tensor([
    [-1., 0., 1.],
    [-1., 0., 1.],
    [-1., 0., 1.],
])
patch = small_image[:3, :3]
manual_value = (patch * edge_kernel).sum()
small_output = F.conv2d(small_image[None, None], edge_kernel[None, None])[0, 0]
print("Hand calculation:", manual_value.item())
print("PyTorch, same position:", small_output[0, 0].item())
assert torch.allclose(manual_value, small_output[0, 0])

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, values, title in zip(axes, [patch, edge_kernel, patch * edge_kernel, small_output],
                            ["Input patch", "Shared kernel", "Elementwise products", "Output feature for the whole image"]):
    matrix_plot(ax, values.numpy(), title)
fig.tight_layout()
plt.show()

### Watch the same weights slide
Press **Play** below, or use the animation slider. Each step fills one output position.
The filter stays the same; the input patch changes. If animation playback is unavailable,
the static figure above contains the same calculation.

In [ ]:
# Plotting helper: no need to study the animation implementation during class.
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
plt.close(fig)

def draw_convolution_step(frame):
    for ax in axes:
        ax.clear()
    r, c = divmod(frame, 3)
    matrix_plot(axes[0], small_image.numpy(), "Input: current 3 x 3 patch")
    axes[0].add_patch(Rectangle((c - .5, r - .5), 3, 3, fill=False, edgecolor="#00a98f", linewidth=3))
    matrix_plot(axes[1], edge_kernel.numpy(), "Same kernel at every position")
    visible = np.full((3, 3), np.nan)
    visible.flat[:frame + 1] = small_output.numpy().flat[:frame + 1]
    axes[2].imshow(visible, cmap="coolwarm", vmin=-3, vmax=3)
    for rr in range(3):
        for cc in range(3):
            if np.isfinite(visible[rr, cc]):
                axes[2].text(cc, rr, f"{visible[rr, cc]:g}", ha="center", va="center")
    axes[2].set(xticks=range(3), yticks=range(3), title=f"Output[{r}, {c}] = {small_output[r, c]:g}")
    fig.tight_layout()

sliding_animation = FuncAnimation(fig, draw_convolution_step, frames=9, interval=700, repeat=True)
display(HTML(sliding_animation.to_jshtml(default_mode="once")))

### Three filters, three different responses · Predict first

The next cell shows a simple image with both horizontal and vertical boundaries, and
three kernels labelled **A, B and C**. Each displayed number is a weight.

Before running the **second** cell, discuss:
1. Which kernel averages its neighborhood and therefore smooths the image?
2. Which detects left-to-right intensity changes (vertical boundaries)?
3. Which detects top-to-bottom intensity changes (horizontal boundaries)?
4. What would each kernel output inside a uniform region, away from the image boundary?

Do the predictions from the weights, not the names. The response plots are in the next cell.

In [ ]:
# First cell: image and kernel weights only. No responses yet.
filter_test_image = torch.zeros(1, 1, 17, 17)
filter_test_image[:, :, 4:13, 5:12] = 1.0
hand_filters = torch.stack([torch.ones(3, 3) / 9, edge_kernel, edge_kernel.T])[:, None]
fig, axes = plt.subplots(1, 4, figsize=(11, 3.1))
axes[0].imshow(filter_test_image[0, 0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Input image")
axes[0].axis("off")
for ax, weights, label in zip(axes[1:], hand_filters[:, 0], ["A", "B", "C"]):
    ax.imshow(weights, cmap="coolwarm", vmin=-1, vmax=1)
    for (r, c), value in np.ndenumerate(weights.numpy()):
        text = "1/9" if label == "A" else f"{value:g}"
        ax.text(c, r, text, ha="center", va="center",
                color="white" if abs(value) > .65 else "black", fontsize=12)
    ax.set_title(f"Kernel {label}")
    ax.set(xticks=[], yticks=[])
fig.tight_layout()
plt.show()

### Reveal the responses
Run the next cell only after making your predictions.
Red and blue denote positive and negative edge responses, respectively.
After the simple image, change `USE_DIGIT` to `True` to apply the same kernels to a digit.
These are still **hand-designed filters**, not the result of training a CNN.

In [ ]:
# Second cell: reveal the outputs, then optionally try a digit.
USE_DIGIT = False
SAMPLE_INDEX = 7  # 0 to 29; used when USE_DIGIT is True.
filter_image = demo_images[SAMPLE_INDEX:SAMPLE_INDEX + 1]  # Also used in the next section.
response_input = filter_image if USE_DIGIT else filter_test_image
hand_responses = F.conv2d(response_input, hand_filters, padding=1)
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(response_input[0, 0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Original")
axes[1].imshow(hand_responses[0, 0], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("A: mean / smoothing")
edge_limit = max(float(hand_responses[0, 1:].abs().max()), 1e-6)
for ax, response, title in zip(axes[2:], hand_responses[0, 1:],
                               ["B: vertical boundaries", "C: horizontal boundaries"]):
    ax.imshow(response, cmap="coolwarm", vmin=-edge_limit, vmax=edge_limit)
    ax.set_title(title)
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()
print("Uniform interior: A returns the constant value; B and C return zero.")

## 3 · Stride and padding: two different design choices

The kernel determines **what local pattern we look for**. Stride and padding determine
**where we apply it and how many outputs we produce**.
Throughout this section: a `3×3` kernel, dilation 1, and zero padding when padding is used.

### 3.1 Why change the stride?

**Stride is the distance between successive window starts**, measured in input pixels.
With stride 1, starts along a row are `0, 1, 2, ...`; with stride 2 they are `0, 2, 4, ...`.
At the end of a row, the window moves down by the same stride and starts a new row.
The kernel still reads all nine values inside each `3×3` window.

- **Stride 1:** sample local patterns densely. This is useful when we want a detailed spatial feature map.
- **Stride 2:** produce a coarser feature map. This is useful when we deliberately downsample,
  reducing activation memory and the number of convolution calculations.
  For sufficiently large images, doubling the stride in both directions gives roughly
  one quarter as many spatial outputs for the same kernel and channels.
- **Trade-off:** fewer window positions means less spatial detail. Small features can be represented
  less reliably. A larger stride is a design choice, not an automatic improvement.

A stride-2 convolution can downsample while applying learned filters. Max pooling also downsamples,
but its local operation is a fixed maximum rather than a learned weighted sum.

**Quick prediction:** does stride 2 change the filter's nine weights, or the places where we use them?

### 3.2 Why add padding?

Without padding, we only use windows that fit completely inside the input.
A `5×5` image with a `3×3` kernel and stride 1 produces a `3×3` output.
Apply another such convolution and it becomes `1×1`:

```text
No padding:         5×5 → 3×3 → 1×1
One zero each side: 5×5 → 5×5 → 5×5     (kernel 3, stride 1)
```

**Padding adds a border before applying the kernel.** `padding=1` adds one row at the top
and bottom and one column on the left and right, so a `5×5` input becomes a `7×7` padded grid.

Why use it?
1. **Control shrinking:** preserve spatial size when stacking layers, rather than losing two rows
   and columns at every `3×3`, stride-1 convolution.
2. **Compute outputs centered at the original boundary pixels:** without padding, a `3×3` window
   cannot be centered at a corner. With padding, the missing neighbors are supplied by the border.
   Boundary pixels do participate without padding, but in fewer windows than interior pixels.

**Trade-off:** added zeros are assumed values, not newly observed image content.
They can influence responses near the edges. Padding controls shape and boundary handling;
it does not recover information outside the image.

**When is padding 0 useful?** When we want every output to depend only on a fully observed input
patch, or when the reduction in size is intentional. Neither choice changes the kernel's parameter count.

**Quick prediction:** can padding 1 preserve size if we also change stride from 1 to 2?

### 3.3 Watch the two choices separately

All three panels below use the **same `5×5` image and the same vertical-edge kernel**:

```text
-1  0  1
-1  0  1
-1  0  1
```

| Compare | What stays fixed? | What to watch |
|---|---|---|
| A → B: add padding | Image, kernel, stride 1 | Extra border positions; output expands from 3×3 to 5×5 |
| B → C: increase stride | Image, kernel, padding 1 | Window starts jump by 2; output shrinks from 5×5 to 3×3 |

**Read the animation:** top row = input and active window; bottom row = outputs computed so far.
Orange marks added zeros, the dashed outline marks the original image, and the green box is the
current `3×3` window. A green dot marks its top-left corner. Output color scales are identical.
Blank output squares have **not yet been computed**; a printed `0` is a computed zero.

The animation starts at the first window. Press **Play**, or use the slider to inspect individual steps.
Each running case computes one output per animation step. A completed case stays on its final state;
this is a teaching animation, not a runtime benchmark.

**Pause and discuss:** compare the first window in A and B. Which original pixel is at its center?
Then compare successive positions in B and C. Does stride 2 necessarily leave gaps between `3×3` windows?

In [ ]:
# Visualization helper. The calculations below also check every animated value against PyTorch.
def make_stride_padding_demo():
    input_grid = small_image.clone()
    shared_kernel = edge_kernel.clone()
    cases = []
    for label, padding, stride in [('A: no padding', 0, 1),
                                    ('B: add padding', 1, 1),
                                    ('C: larger stride', 1, 2)]:
        padded = F.pad(input_grid, (padding, padding, padding, padding))
        expected = F.conv2d(input_grid[None, None], shared_kernel[None, None],
                            padding=padding, stride=stride)[0, 0]
        manual = torch.empty_like(expected)
        for row in range(expected.shape[0]):
            for col in range(expected.shape[1]):
                patch = padded[row * stride:row * stride + 3,
                               col * stride:col * stride + 3]
                manual[row, col] = (patch * shared_kernel).sum()
        torch.testing.assert_close(manual, expected)
        cases.append(dict(label=label, padding=padding, stride=stride,
                          padded=padded.numpy(), output=manual.numpy()))

    fig, axes = plt.subplots(2, 3, figsize=(11, 7))
    fig.subplots_adjust(top=.86, bottom=.07, hspace=.52, wspace=.35)
    fig.suptitle('Same image and kernel | orange: padding | green: active window', fontsize=13, y=.98)
    output_limit = max(max(float(np.abs(case['output']).max()), 1.0) for case in cases)
    response_cmap = plt.get_cmap('coolwarm').copy()
    response_cmap.set_bad('#ededed')

    def draw_step(frame):
        for column, case in enumerate(cases):
            ax_input, ax_output = axes[:, column]
            ax_input.clear()
            ax_output.clear()
            padded, result = case['padded'], case['output']
            padding, stride = case['padding'], case['stride']
            completed = min(frame + 1, result.size)
            row, col = divmod(completed - 1, result.shape[1])
            start_r, start_c = row * stride, col * stride
            side = padded.shape[0]
            ax_input.imshow(padded, cmap='Blues', vmin=0, vmax=1)
            if padding:
                for xy, width, height in [((-.5, -.5), side, padding),
                                           ((-.5, side-padding-.5), side, padding),
                                           ((-.5, padding-.5), padding, 5),
                                           ((side-padding-.5, padding-.5), padding, 5)]:
                    ax_input.add_patch(Rectangle(xy, width, height, color='#efaa55', alpha=.55))
            ax_input.add_patch(Rectangle((padding-.5, padding-.5), 5, 5,
                                         fill=False, linestyle='--', edgecolor='black', linewidth=1.5))
            for (r, c), value in np.ndenumerate(padded):
                ax_input.text(c, r, f'{value:g}', ha='center', va='center', fontsize=9,
                              color='white' if value > .5 else 'black')
            ax_input.add_patch(Rectangle((start_c-.5, start_r-.5), 3, 3,
                                         fill=False, edgecolor='#009b70', linewidth=3))
            ax_input.scatter([start_c], [start_r], s=20, c='#009b70', zorder=5)
            ax_input.set_title(f"{case['label']} | P={padding}, S={stride}\n"
                               f"Window start (row, col): ({start_r}, {start_c})", fontsize=10)
            ax_input.set_xticks(range(side))
            ax_input.set_yticks(range(side))
            ax_input.tick_params(labelsize=8)

            visible = np.full(result.shape, np.nan)
            visible.flat[:completed] = result.flat[:completed]
            ax_output.imshow(np.ma.masked_invalid(visible), cmap=response_cmap,
                              vmin=-output_limit, vmax=output_limit)
            for (r, c), value in np.ndenumerate(visible):
                if np.isfinite(value):
                    ax_output.text(c, r, f'{value:g}', ha='center', va='center', fontsize=10,
                                   color='white' if abs(value) > .65 * output_limit else 'black')
            ax_output.add_patch(Rectangle((col-.5, row-.5), 1, 1,
                                          fill=False, edgecolor='#009b70', linewidth=2.5))
            state = 'complete' if completed == result.size else f'{completed}/{result.size} outputs'
            ax_output.set_title(f"Output {result.shape[0]} x {result.shape[1]} | {state}\n"
                                f"sum(patch * kernel) = {result[row, col]:g}", fontsize=10)
            ax_output.set_xticks(range(result.shape[1]))
            ax_output.set_yticks(range(result.shape[0]))
            ax_output.tick_params(labelsize=8)
        return fig

    frame_count = max(case['output'].size for case in cases)
    return fig, draw_step, frame_count

stride_padding_figure, draw_stride_padding_step, stride_padding_frames = make_stride_padding_demo()
plt.close(stride_padding_figure)  # Display the animation only.
stride_padding_animation = FuncAnimation(stride_padding_figure, draw_stride_padding_step,
                                        frames=stride_padding_frames, interval=650, repeat=False)
display(HTML(stride_padding_animation.to_jshtml(default_mode='once')))

### 3.4 Derive the output size by counting valid window positions

Look along just one axis. After padding, its length is $H+2P$.
A window of length $K$ begins at the left edge and moves right by $S$ at each step.
Its **left-end index** takes the values below; this moving position is not an extra hyperparameter:

$$0,\ S,\ 2S,\ \ldots\quad\text{with left-end index}\le H+2P-K.$$

Therefore the number of positions where the full window fits, including position 0, is

$$H_{\mathrm{out}}=\left\lfloor\frac{H+2P-K}{S}\right\rfloor+1.$$

Use the same calculation for width. The floor discards a last step that would overrun the padded input.
For the animation's `5×5` input and kernel 3:

- A: $P=0,S=1$: window left ends `0, 1, 2` → **3×3** output.
- B: $P=1,S=1$: window left ends `0, 1, 2, 3, 4` on the padded grid → **5×5** output.
- C: $P=1,S=2$: window left ends `0, 2, 4` on the padded grid → **3×3** output.

**A and C have the same output size but compute different windows.**
Equal shapes do not imply equal computations.

For an odd kernel, dilation 1 and **stride 1**, $P=(K-1)/2$ preserves spatial size.
This does not imply that the same padding preserves size with stride 2.

**Your turn (about 60 seconds):** use a `7×7` input and a `3×3` kernel.
Predict the output shape, then run the next cell.

| Padding | Stride | Your output size |
|---:|---:|---|
| 0 | 1 | ? |
| 1 | 1 | ? |
| 1 | 2 | ? |

**★ Extra:** look at one row of an `8×8` input: its pixel indices are `0–7`.
A width-3 window starts at the left edge and moves right by 2 pixels each time, with no padding.
How many positions let the whole window fit? After its left end reaches index 4,
would moving it to index 6 still fit? Write out the three indices it would cover.

In [ ]:
def output_size(size, kernel, stride=1, padding=0):
    return (size + 2 * padding - kernel) // stride + 1

shape_input = torch.arange(49, dtype=torch.float32).reshape(1, 1, 7, 7)
shape_kernel = edge_kernel[None, None]
input_h, input_w = shape_input.shape[-2:]
kernel_h, kernel_w = shape_kernel.shape[-2:]
print(f"Input shape [B, C_in, H, W]: {tuple(shape_input.shape)}")
print(f"Kernel size: {kernel_h} x {kernel_w}; weight shape [C_out, C_in, K_H, K_W]: {tuple(shape_kernel.shape)}")
print("Dilation = 1; no bias. Only padding and stride change below.")
for padding, stride in [(0, 1), (1, 1), (1, 2)]:
    result = F.conv2d(shape_input, shape_kernel, padding=padding, stride=stride)
    predicted_hw = (output_size(input_h, kernel_h, stride, padding),
                    output_size(input_w, kernel_w, stride, padding))
    print(f"P={padding}, S={stride}: predicted output [H, W] = {predicted_hw}; "
          f"actual output [B, C_out, H, W] = {tuple(result.shape)}")
    assert tuple(result.shape[-2:]) == predicted_hw

# Edit these two values and predict the result before rerunning.
STRIDE, PADDING = 2, 1
changed_response = F.conv2d(filter_image, edge_kernel[None, None], stride=STRIDE, padding=PADDING)
print(f"Digit input [B, C, H, W] = {tuple(filter_image.shape)}; "
      f"kernel = {tuple(edge_kernel.shape)}, P={PADDING}, S={STRIDE} "
      f"-> output [B, C_out, H, W] = {tuple(changed_response.shape)}")
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(filter_image[0, 0], cmap="gray")
axes[0].set_title("Input: 28 x 28")
lim = max(float(changed_response.abs().max()), 1e-6)
axes[1].imshow(changed_response[0, 0], cmap="coolwarm", vmin=-lim, vmax=lim)
axes[1].set_title(f"Response: {changed_response.shape[-2]} x {changed_response.shape[-1]}")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()

### Multiple input AND output channels: one full filter per output channel

For a normal convolution (`groups=1`), **every output channel reads all input channels**.
The output channels use different learned filters. They are not copies of the input channels,
and their number does not have to equal the input channel count.

Start with **2 input channels and 2 output channels**, using `2×2` spatial kernels.
To keep the arithmetic small, each input channel is a `3×3` matrix; stride is 1 and padding is 0.

For each output channel, a complete filter contains **two `2×2` slices**: one for each input channel.
Here `W[o, c]` means the slice connecting input channel `c` to output channel `o`:

| | Read input X₀ | Read input X₁ | Add once per output position |
|---|---|---|---|
| Output Y₀ | W[0, 0] | W[0, 1] | bias b₀ |
| Output Y₁ | W[1, 0] | W[1, 1] | bias b₁ |

**At one spatial position:**
1. For output 0, multiply its first weight slice by the patch in input 0 and sum the products.
2. Do the same with its second weight slice and the patch in input 1.
3. Add these two contributions and **one bias** → one value in output channel 0.
4. Repeat using output 1's own weight slices and bias → one value in output channel 1.

The two output values are kept in separate channels. **Sum over input channels; stack output channels.**
The channel axis is not another sliding direction: the window slides over height and width while
reading all input channels at the same spatial location.

The next cell displays the inputs and all four weight slices.
Try calculating the top-left value of each output before revealing the result.

In [ ]:
# Two input channels. No batch dimension yet: [C_in=2, H=3, W=3].
channel_inputs = torch.tensor([
    [[1., 2., 0.], [0., 1., 3.], [2., 0., 1.]],
    [[0., 1., 2.], [2., 1., 0.], [1., 3., 2.]],
])
# [C_out=2, C_in=2, K_H=2, K_W=2]
channel_weights = torch.tensor([
    [ [[1., 0.], [0., 1.]], [[0., 1.], [1., 0.]] ],  # Output 0 reads both inputs.
    [ [[1., -1.], [0., 0.]], [[0., 0.], [1., 1.]] ], # Output 1 reads both inputs.
])
channel_biases = torch.tensor([0.5, -1.0])
fig, axes = plt.subplots(2, 3, figsize=(10, 5.5))
for c in range(2):
    matrix_plot(axes[c, 0], channel_inputs[c].numpy(), f"Input channel X{c}", limit=3)
    for o in range(2):
        matrix_plot(axes[c, o+1], channel_weights[o, c].numpy(),
                    f"To output {o}: W[{o}, {c}]", limit=1)
fig.suptitle("Each output column has its own filter spanning BOTH input channels", fontsize=13)
fig.tight_layout()
plt.show()
print("Input batch shape:", tuple(channel_inputs[None].shape))
print("Weights [C_out, C_in, K_H, K_W]:", tuple(channel_weights.shape))
print("Biases [b0, b1]:", channel_biases.tolist())

### Reveal one position, then slide across the whole image

At the **top-left output position**, the input patches are the top-left `2×2` squares.
For output 0:

$$Y_0[0,0]=\underbrace{(1\cdot1+2\cdot0+0\cdot0+1\cdot1)}_{X_0\text{ contribution}=2}
+\underbrace{(0\cdot0+1\cdot1+2\cdot1+1\cdot0)}_{X_1\text{ contribution}=3}
+0.5=5.5.$$

For output 1, use its **different** weights with the **same** input patches:

$$Y_1[0,0]=\underbrace{(1\cdot1+2\cdot(-1)+0\cdot0+1\cdot0)}_{X_0\text{ contribution}=-1}
+\underbrace{(0\cdot0+1\cdot0+2\cdot1+1\cdot1)}_{X_1\text{ contribution}=3}
-1=1.$$

Now slide the two complete filters to all valid positions.
Each input-to-output connection produces a temporary `2×2` contribution map.
**Within each row below, add the two maps element by element, then add that output's bias.**
Keep the two rows' results as separate output channels. No sum is taken between output channels.

In [ ]:
# Compute each input-channel contribution separately so the channel sum is visible.
channel_contributions = torch.empty(2, 2, 2, 2)  # [C_out, C_in, H_out, W_out]
for o in range(2):
    for c in range(2):
        channel_contributions[o, c] = F.conv2d(
            channel_inputs[c][None, None], channel_weights[o, c][None, None]
        )[0, 0]
channel_outputs_manual = channel_contributions.sum(dim=1) + channel_biases[:, None, None]
channel_outputs_torch = F.conv2d(channel_inputs[None], channel_weights, bias=channel_biases)
torch.testing.assert_close(channel_outputs_manual, channel_outputs_torch[0])

for o in range(2):
    values = channel_contributions[o, :, 0, 0]
    print(f"Y{o}[0,0] = {values[0].item():g} + {values[1].item():g} "
          f"+ ({channel_biases[o].item():g}) = {channel_outputs_manual[o,0,0].item():g}")
fig, axes = plt.subplots(2, 3, figsize=(11, 5.8))
limit = max(float(channel_contributions.abs().max()), float(channel_outputs_manual.abs().max()))
for o in range(2):
    matrix_plot(axes[o, 0], channel_contributions[o, 0].numpy(),
                f"Output {o}: contribution from X0", limit=limit)
    matrix_plot(axes[o, 1], channel_contributions[o, 1].numpy(),
                f"+ contribution from X1", limit=limit)
    matrix_plot(axes[o, 2], channel_outputs_manual[o].numpy(),
                f"+ bias ({channel_biases[o].item():g}) = Y{o}", limit=limit)
fig.suptitle("Sum ACROSS each row; keep the two output channels separate", fontsize=13)
fig.tight_layout()
plt.show()
print("PyTorch output [B, C_out, H_out, W_out]:", tuple(channel_outputs_torch.shape))
print("Two 2 x 2 output maps, stacked along the channel axis; no activation applied yet.")

### Generalize to RGB input and eight output channels

`nn.Conv2d(3, 8, kernel_size=3, padding=1)` follows exactly the same pattern:

- **3 input channels:** each complete output filter has three different `3×3` weight slices.
- **8 output channels:** learn eight such filters, each with its own bias.
- Weight tensor: **`[8, 3, 3, 3]`**. This is 8 complete filters, or 24 two-dimensional slices.
- Bias tensor: **`[8]`**, not `[8, 3]`. One bias is added after combining the three input contributions.
- For each image, each filter produces one feature map. Stack eight maps to form the output tensor.

For output channel $o$ and output position $(r,s)$, at stride 1 and no padding for simplicity:

$$Y_o[r,s]=b_o+\sum_{c=0}^{C_{\mathrm{in}}-1}\sum_{u=0}^{K_H-1}\sum_{v=0}^{K_W-1}
W[o,c,u,v]X[c,r+u,s+v].$$

The sums combine the spatial patch **and input channels**. The output channel $o$ is fixed during
this calculation; choose a different $o$ to use a different filter.
The same filters and biases are reused across spatial positions and all images in a batch.

**Check your understanding:** with 3 input channels and 8 output channels, do we produce
24 final feature maps or 8? Where do the contributions from the three inputs get combined?

In [ ]:
rgb_layer = nn.Conv2d(3, 8, kernel_size=3, padding=1)
rgb_batch = torch.zeros(2, 3, 32, 32)
print("Input [B, C_in, H, W]:", tuple(rgb_batch.shape))
print("Weights [C_out, C_in, K_H, K_W]:", tuple(rgb_layer.weight.shape))
print("Biases [C_out]:", tuple(rgb_layer.bias.shape))
print("Output [B, C_out, H_out, W_out]:", tuple(rgb_layer(rgb_batch).shape))
print("Each of the 8 output maps sums 3 input-channel contribution maps, then adds one bias.")

## 4 · Count parameters: what is shared?

For an ordinary convolution (`groups=1`) with bias:

$$\#\text{parameters}=C_{\mathrm{out}}(C_{\mathrm{in}}K_HK_W+1).$$

For `Conv2d(3, 8, 3)`: $8(3\cdot3\cdot3+1)=224$.
The same 224 numbers are used for every image and every spatial position.

**Discuss before running:**
1. If the image changes from `32×32` to `64×64`, does the parameter count change?
2. If batch size doubles, does the parameter count change?
3. If output channels double, what happens to parameters and output channels?

Larger images and batches increase activations and computation, even when the parameters stay fixed.

In [ ]:
for batch, side in [(2, 32), (2, 64), (4, 32)]:
    result = rgb_layer(torch.zeros(batch, 3, side, side))
    print(f"Input {(batch, 3, side, side)} -> {tuple(result.shape)}, parameters = {parameter_count(rgb_layer)}")

more_channels = nn.Conv2d(3, 16, 3, padding=1)
print("16 output channels:", parameter_count(more_channels), "parameters")

# Compare layers that produce the SAME output tensor size from a 32 x 32 RGB image.
# Count a hypothetical dense layer algebraically; do not allocate its huge weight matrix.
dense_input_units = 3 * 32 * 32
dense_output_units = 8 * 32 * 32
dense_parameters = dense_output_units * (dense_input_units + 1)
print(f"Fully connected, 3072 -> 8192 outputs: {dense_parameters:,} parameters")
print(f"Conv2d, same output volume:           {parameter_count(rgb_layer):,} parameters")

The dense comparison isolates the effect of local connections and weight sharing for the
same output size. A practical MLP might instead choose a much smaller hidden layer.
This count alone does not prove one whole model is always smaller or more accurate.

<details><summary>Reveal the discussion answers</summary>
1. The convolution still has 224 parameters; with padding 1 and stride 1, four times as many spatial outputs are computed.
2. It still has 224 parameters; the same model processes each sample.
3. It has 448 parameters and 16 output channels.
</details>

## 5 · Convolution beyond images
### 5.1 A large smooth wave and a smaller sharp event

A 1D convolution slides along **time or sequence position**. Consider a synthetic sensor signal:
- a slow sine wave with amplitude **3**, plus a smaller smooth oscillation;
- a brief added spike with amplitude **1**, smaller than the main wave;
- optionally, a small amount of noise shared by both examples.

We compare the same baseline **without** and **with** the spike.
The default spike is away from the largest wave peak, so both signals have the same global maximum.
These are teaching signals, not real medical measurements.

**Predict:** can a rule based only on `max(signal)` distinguish this pair?
What differs locally even though the spike is not the tallest point?

In [ ]:
SIGNAL_LENGTH = 100
SPIKE_POSITION = 50  # Default keeps the spike below the existing wave peak.
SPIKE_AMPLITUDE = 1.0
NOISE_STD = 0.0      # After the clean example, try 0.03 or 0.15.
assert 2 <= SPIKE_POSITION <= SIGNAL_LENGTH - 3
assert SPIKE_AMPLITUDE >= 0 and NOISE_STD >= 0
signal_time = torch.arange(SIGNAL_LENGTH, dtype=torch.float32)
noise_generator = torch.Generator().manual_seed(42)
smooth_signal = (3.0 * torch.sin(2 * torch.pi * signal_time / 100)
                 + 0.4 * torch.sin(2 * torch.pi * signal_time / 40 + 0.5))
baseline_signal = smooth_signal + NOISE_STD * torch.randn(SIGNAL_LENGTH, generator=noise_generator)
sensor_signal = baseline_signal.clone()
sensor_signal[SPIKE_POSITION] += SPIKE_AMPLITUDE
signal_batch = sensor_signal[None, None]  # [B=1, C=1, L=100]

fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True, sharey=True)
for ax, values, title in zip(axes, [baseline_signal, sensor_signal],
                             ["Without the spike", "With a smaller, sharp spike"]):
    ax.plot(signal_time, values)
    ax.axhline(float(baseline_signal.max()), color="gray", linestyle="--", linewidth=1,
               label="Baseline global maximum")
    ax.set(ylabel="Signal value", title=title)
axes[0].legend(loc="lower left", fontsize=9)
axes[1].annotate(f"Added spike: +{SPIKE_AMPLITUDE:g}",
                 xy=(SPIKE_POSITION, float(sensor_signal[SPIKE_POSITION])),
                 xytext=(SPIKE_POSITION + 8, 2.3),
                 arrowprops={"arrowstyle": "->", "color": "#2878a5"}, color="#2878a5")
axes[1].set_xlabel("Time index")
fig.tight_layout()
plt.show()
print(f"Raw maximum without spike: {baseline_signal.max().item():.4f}")
print(f"Raw maximum with spike:    {sensor_signal.max().item():.4f}")
print(f"Main sine amplitude: 3.0 | added spike amplitude: {SPIKE_AMPLITUDE:g}")

### Detect local change instead of absolute height

Apply the hand-designed kernel `[-1, 0, 1]`:

$$y_t=-x_{t-1}+x_{t+1}.$$

A large but slowly changing wave can have a small response. A smaller but abrupt event can have
an appreciable response. The kernel compares neighbors instead of asking whether the raw signal is high.

**Predict the response before running.** We use no padding, so each output uses a full three-sample
window and is plotted at that window's center. The spike produces responses on its two sides;
the strongest response need not be at the spike's center.

In [ ]:
change_kernel = torch.tensor([[[-1., 0., 1.]]])
normal_response = F.conv1d(baseline_signal[None, None], change_kernel)[0, 0]
signal_response = F.conv1d(signal_batch, change_kernel)[0, 0]
response_centers = signal_time[1:-1]
fig, axes = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True)
axes[0].plot(response_centers, normal_response, label="Without spike")
axes[0].plot(response_centers, signal_response, linestyle="--", label="With spike")
axes[0].set(ylabel="Signed response", title="Same shared kernel, two signals")
axes[1].plot(response_centers, normal_response.abs(), label="Without spike")
axes[1].plot(response_centers, signal_response.abs(), linestyle="--", label="With spike")
axes[1].set(xlabel="Time index (window center)", ylabel="Absolute response")
for ax in axes:
    ax.axvline(SPIKE_POSITION, color="gray", linestyle=":", linewidth=1)
    ax.legend(loc="upper right")
fig.tight_layout()
plt.show()
print(f"Maximum absolute response without spike: {normal_response.abs().max().item():.4f}")
print(f"Maximum absolute response with spike:    {signal_response.abs().max().item():.4f}")
print("Input:", tuple(signal_batch.shape), "| Conv1d output:", (1, 1, len(signal_response)))

**What changed?** A maximum over the *raw signal* measures height. A maximum over the
*absolute convolution response* summarizes local changes. The latter uses the local structure
computed by the filter first.

Try `NOISE_STD = 0.03`, then `0.15`, and rerun both signal cells.
Does the background response grow? Difference filters also respond to rapid noise;
this example is not a universal anomaly detector. The default signal pair was chosen to separate
height from abruptness, not to establish a real-world accuracy result.

We have used a **fixed 1D convolution**, not yet a trained 1D CNN. Part B will show how
learned filters can be combined with other layers for a classification task.

### 5.2 What do 1D, 2D and 3D refer to?

| Layer | Sliding dimensions | Batched input layout | Example |
|---|---|---|---|
| `Conv1d` | Sequence position / time | `[B, C, L]` | Audio waveform; sensor signals |
| `Conv2d` | Height and width | `[B, C, H, W]` | Images |
| `Conv3d` | Depth, height and width | `[B, C, D, H, W]` | CT volume; video with time as D |

**Channels are not the sliding dimensions.** An RGB image usually uses **2D** convolution
although it has three channels. Three synchronized sensors can be three input channels to a
**1D** convolution.

### What is a CT volume?
A CT scan can be represented as an ordered stack of 2D cross-sectional images of the body.
One slice has shape `[H, W]`; stacking slices along a third spatial axis gives `[D, H, W]`.
An element is a **voxel** (a small volume element), rather than a 2D pixel.
A `3×3×3` kernel combines a local `3×3` region across three neighboring slices.

In a CT volume, D denotes **spatial depth**. In video, the analogous axis can denote **time**.
The diagram below uses synthetic slices solely to illustrate the array layout; it is not a real CT scan.

In [ ]:
# Schematic synthetic slices: show how local regions span the depth axis.
yy, xx = np.mgrid[-1:1:40j, -1:1:40j]
fig, axes = plt.subplots(1, 4, figsize=(11, 3.0))
for i, ax in enumerate(axes[:3]):
    radius = [0.62, 0.72, 0.66][i]
    slice_image = ((xx**2 + (yy / .85)**2) < radius**2).astype(float) * .65
    slice_image[((xx-.12)**2 + (yy+.12)**2) < .12**2] = 1.0
    ax.imshow(slice_image, cmap="gray", vmin=0, vmax=1)
    ax.add_patch(Rectangle((18.5, 18.5), 3, 3, fill=False, edgecolor="#00a98f", linewidth=2))
    ax.set_title(f"Slice d = {i}")
    ax.axis("off")
axes[3].axis("off")
axes[3].text(.5, .55, "Stack along depth D\n\n[D, H, W]\n\n3 slices × 3 rows × 3 columns\n= 27 local voxels\nper input channel",
             ha="center", va="center", fontsize=10, color=PURPLE)
fig.suptitle("Schematic volume: neighboring slices and corresponding 3 × 3 regions", fontsize=12)
fig.tight_layout()
plt.show()

# Part B · From convolution operations to a CNN

In Part A we chose kernel values ourselves. **For digit recognition, how can the model learn useful kernels?**

A convolutional layer stores its weights and biases as trainable parameters. A classification loss
measures the model's error; backpropagation computes gradients; the optimizer updates these parameters,
just as it updated the linear layers last week. We do not supply target edge maps for each hidden layer.

A complete CNN combines these learned convolutions with other operations and an output layer:

```text
image → learned convolution → ReLU → pooling → more features → class scores
```

The convolution is one operation inside this model. ReLU and pooling do different jobs.
We will load a model trained before class; the full training code is in the appendix.

## 6 · ReLU and max pooling
### 6.1 ReLU: a function applied to each value

$$\operatorname{ReLU}(x)=\max(0,x).$$

Negative inputs become zero; positive inputs pass through unchanged. ReLU introduces nonlinearity,
as in last week's MLP. It changes neither the tensor shape nor the number of channels and has
no learned parameters. First look at one number, then at a whole feature map.

In [ ]:
relu_x = torch.linspace(-3, 3, 301)
relu_points = torch.tensor([-2., -1., 0., 1., 2.])
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.axhline(0, color="gray", linewidth=.8)
ax.axvline(0, color="gray", linewidth=.8)
ax.plot(relu_x, F.relu(relu_x), linewidth=3, color=PURPLE, label="ReLU(x) = max(0, x)")
ax.scatter(relu_points, F.relu(relu_points), s=40, c="#2878a5", zorder=4)
for x, y in zip(relu_points.tolist(), F.relu(relu_points).tolist()):
    ax.annotate(f"({x:g}, {y:g})", (x, y), xytext=(0, 11), textcoords="offset points",
                ha="center", fontsize=10)
ax.text(-2.75, 1.45, "Negative input\n→ zero output", color=PURPLE)
ax.text(.25, 2.65, "Positive input\n→ unchanged", color=PURPLE)
ax.set(xlabel="Input x", ylabel="Output ReLU(x)", xlim=(-3, 3), ylim=(-.4, 3.4))
ax.grid(alpha=.15)
fig.tight_layout()
plt.show()

In [ ]:
activation_example = torch.tensor([
    [-1., 2., 0., 3.],
    [4., -2., 1., 5.],
    [2., 1., -3., -1.],
    [0., 6., -2., -4.],
])
relu_example = F.relu(activation_example)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for ax, values, title in zip(axes, [activation_example, relu_example],
                            ["Before ReLU", "After ReLU: same shape"]):
    matrix_plot(ax, values.numpy(), title, limit=6)
fig.tight_layout()
plt.show()
print("ReLU parameters:", parameter_count(nn.ReLU()))

### 6.2 Max pooling: select a maximum from each local window

A `2×2` max-pooling window with stride 2 takes four values and emits one maximum.
For a `4×4` feature map, four windows produce a `2×2` output.
Pooling reduces spatial resolution and discards some information.

**Predict the four output values before pressing Play.** The green box marks the current window;
orange circles mark the maximum value(s); the corresponding output is filled below.
Tied maxima give the same output value. Pooling has no learned weights.
It operates **independently within each channel**, so the channel count does not change.

In [ ]:
# Four-step max-pooling animation; no separate static output.
def make_pooling_demo():
    values = relu_example.numpy()
    expected = F.max_pool2d(relu_example[None, None], 2, stride=2)[0, 0].numpy()
    fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.8))
    fig.subplots_adjust(top=.80, wspace=.4, bottom=.13)

    def draw(frame):
        for ax in axes:
            ax.clear()
        r, c = divmod(frame, 2)
        top, left = 2*r, 2*c
        patch = values[top:top+2, left:left+2]
        maximum = float(patch.max())
        assert maximum == float(expected[r, c])
        matrix_plot(axes[0], values, "Input: 4 x 4 (one channel)", limit=6)
        axes[0].add_patch(Rectangle((left-.5, top-.5), 2, 2, fill=False,
                                   edgecolor="#009b70", linewidth=3))
        for pr, pc in np.argwhere(patch == maximum):
            axes[0].scatter([left+pc], [top+pr], s=360, facecolors="none",
                            edgecolors="#e29a17", linewidths=3)
        visible = np.full((2, 2), np.nan)
        visible.flat[:frame+1] = expected.flat[:frame+1]
        cmap = plt.get_cmap("coolwarm").copy()
        cmap.set_bad("#eeeeee")
        axes[1].imshow(np.ma.masked_invalid(visible), cmap=cmap, vmin=-6, vmax=6)
        for (rr, cc), value in np.ndenumerate(visible):
            if np.isfinite(value):
                axes[1].text(cc, rr, f"{value:g}", ha="center", va="center", fontsize=13,
                             color="white" if value>3.9 else "black")
        axes[1].add_patch(Rectangle((c-.5, r-.5), 1, 1, fill=False,
                                   edgecolor="#009b70", linewidth=3))
        axes[1].set(title=f"Output: 2 x 2 | {frame+1}/4 computed", xticks=[0,1], yticks=[0,1])
        fig.suptitle(f"Step {frame+1}: max{tuple(int(v) for v in patch.flat)} = {maximum:g}", fontsize=13)
        return fig
    return fig, draw

pooling_figure, draw_pooling_step = make_pooling_demo()
plt.close(pooling_figure)
pooling_animation = FuncAnimation(pooling_figure, draw_pooling_step, frames=4, interval=1100, repeat=False)
display(HTML(pooling_animation.to_jshtml(default_mode="once")))
print("Pooling parameters:", parameter_count(nn.MaxPool2d(2)))

## 7 · Receptive field: trace one output back to the input

The **theoretical receptive field** of an activation is the input region that can affect it
through the network's connections. It describes possible dependencies, not how strongly the trained
model actually uses each pixel.

In this example both convolutions have kernel `3×3`, stride 1 and dilation 1, with no padding.
We choose an interior output and omit ReLU from the diagram to focus on connectivity:

```text
7×7 input → first convolution: 5×5 feature map → second convolution: 3×3 feature map
```

Watch the animation **from right to left**:
1. Choose the center value in the second feature map.
2. Highlight the `3×3` region it reads in the first feature map.
3. Visit those nine first-layer values and highlight each one's `3×3` input region.
4. Their combined input region is **`5×5`**.

Adding ReLU between the layers adds nonlinearity but does not enlarge this theoretical region.
A particular activation can have weaker or zero dependence on some pixels because of learned
weights or activation values.

In [ ]:
# Trace dependencies backwards. Colors distinguish the active patch from the accumulated union.
def make_receptive_field_demo():
    fig, axes = plt.subplots(1, 3, figsize=(11, 4))
    fig.subplots_adjust(top=.76, bottom=.15, wspace=.32)
    fig.text(.5,.04,"Trace dependencies from right to left  ←     (forward computation goes left to right)",
             ha="center",fontsize=10,color=PURPLE)
    neighbors = [(r, c) for r in range(1, 4) for c in range(1, 4)]

    def grid(ax, side, title):
        ax.clear()
        ax.imshow(np.ones((side, side)), cmap="gray", vmin=0, vmax=1)
        ax.set(xticks=range(side), yticks=range(side), title=title)
        ax.set_xticks(np.arange(-.5, side, 1), minor=True)
        ax.set_yticks(np.arange(-.5, side, 1), minor=True)
        ax.grid(which="minor", color="#cccccc", linewidth=.8)
        ax.tick_params(which="minor", bottom=False, left=False)

    def draw(frame):
        for ax, side, title in zip(axes, [7,5,3], ["Input: 7 x 7", "First feature map: 5 x 5",
                                                 "Second feature map: 3 x 3"]):
            grid(ax, side, title)
        axes[2].add_patch(Rectangle((.5,.5),1,1,facecolor=PURPLE,alpha=.7))
        axes[2].text(1,1,"target",ha="center",va="center",color="white",fontsize=9)
        if frame >= 1:
            axes[1].add_patch(Rectangle((.5,.5),3,3,facecolor="#2878a5",alpha=.18,
                                        edgecolor="#2878a5",linewidth=2))
        visited = min(max(frame-1, 0), 9)
        union = np.zeros((7,7),dtype=bool)
        for r,c in neighbors[:visited]:
            union[r:r+3,c:c+3] = True
            axes[1].scatter([c],[r],s=35,c=PURPLE)
        for r,c in np.argwhere(union):
            axes[0].add_patch(Rectangle((c-.5,r-.5),1,1,facecolor=PURPLE,alpha=.25,edgecolor="none"))
        if 2 <= frame <= 10:
            r,c=neighbors[visited-1]
            axes[1].add_patch(Rectangle((c-.5,r-.5),1,1,fill=False,edgecolor="#009b70",linewidth=3))
            axes[0].add_patch(Rectangle((c-.5,r-.5),3,3,fill=False,edgecolor="#009b70",linewidth=3))
        if frame == 0:
            caption="1. Select one second-layer output (right)."
        elif frame == 1:
            caption="2. It reads these nine first-layer values (middle)."
        elif frame <= 10:
            caption=f"3. Trace first-layer value {visited}/9: green = its input patch; purple = union so far."
        else:
            assert union.sum()==25
            axes[0].add_patch(Rectangle((.5,.5),5,5,fill=False,edgecolor=PURPLE,linewidth=3))
            caption="4. The union is 5 x 5: two 3 x 3 convolutions expand the receptive field."
        fig.suptitle(caption,fontsize=12,y=.96)
        return fig
    return fig,draw

receptive_field_figure, draw_receptive_field_step = make_receptive_field_demo()
plt.close(receptive_field_figure)
receptive_field_animation = FuncAnimation(receptive_field_figure, draw_receptive_field_step,
                                         frames=12, interval=1000, repeat=False)
display(HTML(receptive_field_animation.to_jshtml(default_mode="once")))

## 8 · Open up a small CNN

Our network uses two convolution blocks and one final linear classifier:

| Stage | Output shape | Parameters |
|---|---|---:|
| Input | `[B, 1, 28, 28]` | — |
| Conv `1→8`, kernel 3, padding 1 + ReLU | `[B, 8, 28, 28]` | 80 |
| MaxPool 2 | `[B, 8, 14, 14]` | 0 |
| Conv `8→16`, kernel 3, padding 1 + ReLU | `[B, 16, 14, 14]` | 1,168 |
| MaxPool 2 | `[B, 16, 7, 7]` | 0 |
| Flatten each sample | `[B, 784]` | 0 |
| Linear `784→10` | `[B, 10]` | 7,850 |

**Your turn:** explain why the classifier needs $16\cdot7\cdot7=784$ inputs.
The batch dimension is preserved throughout.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(nn.Flatten(start_dim=1), nn.Linear(16 * 7 * 7, 10))

    def forward(self, x):
        return self.classifier(self.features(x))

torch.manual_seed(42)
cnn = TinyCNN()
print(cnn)
print("Total parameters:", parameter_count(cnn))
assert parameter_count(cnn) == 9098

### Load the TA's trained model
The expected checkpoint is produced by the **After-class training** section below.
It stores weights, preprocessing information, actual learning curves, held-out test metrics,
and a small gallery. This cell does not train.

After adding the checkpoint, rerun this cell and the subsequent cells in Section 8.
Re-running the model-definition cell above resets the model; reload the checkpoint afterwards.

In [ ]:
ARCHITECTURE_ID = "tinycnn_1_8_16_k3_pool2_v1"
PREPROCESSING_ID = "uint8_div255_no_normalization"

def load_teaching_checkpoint(path):
    bundle = torch.load(path, map_location="cpu", weights_only=True)
    if bundle.get("architecture") != ARCHITECTURE_ID:
        raise ValueError("Checkpoint architecture differs. Use the training appendix in this notebook.")
    if bundle.get("preprocessing") != PREPROCESSING_ID:
        raise ValueError("Checkpoint preprocessing differs from this notebook.")
    model = TinyCNN()
    model.load_state_dict(bundle["model_state_dict"], strict=True)
    model.eval()
    return model, bundle

checkpoint = None
if CHECKPOINT_PATH.exists():
    cnn, checkpoint = load_teaching_checkpoint(CHECKPOINT_PATH)
    model_status = "PRETRAINED"
    print("Loaded:", CHECKPOINT_PATH)
    print("Training configuration:", checkpoint.get("config", {}))
else:
    torch.manual_seed(42)
    cnn = TinyCNN().eval()
    model_status = "UNTRAINED"
    print("No checkpoint found: using a random model for shapes and mechanics only.")
    print("Trained predictions, performance claims and learning curves are skipped.")

In [ ]:
with torch.no_grad():
    values = demo_images[:2]
    print("Input".ljust(20), tuple(values.shape))
    for layer in list(cnn.features) + list(cnn.classifier):
        values = layer(values)
        print(type(layer).__name__.ljust(20), tuple(values.shape))
print("Each final row contains 10 logits (class scores), not 10 labels.")

### Same image, different intermediate representations
Choose an image and a first-layer output channel. Inspect:
**input → convolution + bias → ReLU → pooling**.
The convolution and ReLU panels use the same numerical color scale, so you can see which
negative values disappear. The colorbar reports activation values, not pixel intensities.

**Discuss:** which locations respond strongly? What does pooling remove?
If the checkpoint is absent, these are random features, not learned ones.

In [ ]:
INSPECT_INDEX = 7     # 0 to 29
INSPECT_CHANNEL = 0   # 0 to 7
inspect_image = demo_images[INSPECT_INDEX:INSPECT_INDEX + 1]
with torch.no_grad():
    conv_values = cnn.features[0](inspect_image)
    relu_values = cnn.features[1](conv_values)
    pooled_values = cnn.features[2](relu_values)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.3), constrained_layout=True)
axes[0].imshow(inspect_image[0, 0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"Input: digit {int(demo_labels[INSPECT_INDEX])}")
limit = max(float(conv_values[0, INSPECT_CHANNEL].abs().max()), 1e-6)
for ax, tensor, name in zip(axes[1:], [conv_values, relu_values, pooled_values],
                            ["Conv + bias", "ReLU", "MaxPool"]):
    im = ax.imshow(tensor[0, INSPECT_CHANNEL], cmap="coolwarm", vmin=-limit, vmax=limit)
    ax.set_title(f"{name}: {tensor.shape[-2]} x {tensor.shape[-1]}")
for ax in axes:
    ax.axis("off")
fig.colorbar(im, ax=axes[1:], shrink=.65, label="Activation")
fig.suptitle(f"{model_status} model | output channel {INSPECT_CHANNEL}")
plt.show()

### Second-layer feature maps across digits 0–9

Now compare **one example of each digit** at the second convolutional layer.
We display activations **after the second ReLU and before the second max pooling**:

```text
[10, 1, 28, 28]
→ Conv1 + ReLU + Pool1
→ [10, 8, 14, 14]
→ Conv2 + ReLU
→ [10, 16, 14, 14]   ← inspect here
```

Unlike the first layer, the second convolution combines **eight input feature channels**.
Each of its 16 filters has shape `[8, 3, 3]`; it is not a single `3×3` image kernel.
Its responses combine local patterns extracted by the first layer.

**Read the galleries:** columns are digits 0–9; the first row shows the original images;
each remaining row shows one fixed second-layer channel. All 16 channels are shown in four
manageable groups. Use `CHANNELS_TO_SHOW` to focus on a few channels during class.

Each channel uses the **same color scale across all ten digits**, with its own colorbar on the right.
Compare brightness **within a row**. Different rows can have different scales; their brightness
alone does not compare absolute response strength. The maps are `14×14`, so they do not match
original-image pixel coordinates one-to-one.

**Look before explaining:**
- In one row, which spatial regions respond strongly? Do similar stroke patterns in different
  digits produce similar responses?
- Compare the examples of 0, 6, 8 and 9, then 1, 4 and 7. What similarities or differences can
  you point to in the maps? It is also valid to find no obvious interpretable pattern.
- Change `EXAMPLE_PER_DIGIT` from 0 to 1 or 2 and rerun. Does your observation survive a different
  handwriting example?

These are examples, not class-average maps. One example does not establish a property of an entire
class. A channel is not automatically a detector for one digit, and a bright region is not by itself
an explanation of the final classification. If the title says **UNTRAINED**, do not interpret the
responses as features learned from data.

In [ ]:
# Compare the SAME second-layer channels across one image from EVERY class.
EXAMPLE_PER_DIGIT = 0               # 0, 1 or 2: three embedded examples of each digit.
CHANNELS_TO_SHOW = list(range(16))  # For a short class view, try [0, 3, 8, 12].
CHANNELS_PER_FIGURE = 4

if EXAMPLE_PER_DIGIT not in (0, 1, 2):
    raise ValueError("EXAMPLE_PER_DIGIT must be 0, 1 or 2.")
if not CHANNELS_TO_SHOW or any(ch not in range(16) for ch in CHANNELS_TO_SHOW):
    raise ValueError("Choose one or more channel indices from 0 to 15.")
comparison_indices = [
    torch.where(demo_labels == digit)[0][EXAMPLE_PER_DIGIT].item()
    for digit in range(10)
]
comparison_images = demo_images[comparison_indices]
with torch.no_grad():
    second_layer_maps = cnn.features[:5](comparison_images)  # Stop after Conv2 + ReLU.
assert tuple(second_layer_maps.shape) == (10, 16, 14, 14)
assert torch.equal(demo_labels[comparison_indices], torch.arange(10))
print(f"{model_status} | example {EXAMPLE_PER_DIGIT + 1} of each digit")
print("Second convolution + ReLU:", tuple(second_layer_maps.shape))
print("Same scale across each row; separate scales between channels.")

for start in range(0, len(CHANNELS_TO_SHOW), CHANNELS_PER_FIGURE):
    selected_channels = CHANNELS_TO_SHOW[start:start + CHANNELS_PER_FIGURE]
    rows = 1 + len(selected_channels)
    fig = plt.figure(figsize=(14.5, 1.45 * rows + .55), constrained_layout=True)
    grid = fig.add_gridspec(rows, 11, width_ratios=[1] * 10 + [.13])
    fig.suptitle(
        f"{model_status} | Conv2 + ReLU | channels {', '.join(map(str, selected_channels))} | "
        f"example {EXAMPLE_PER_DIGIT + 1} per digit",
        fontsize=13,
    )
    for digit in range(10):
        ax = fig.add_subplot(grid[0, digit])
        ax.imshow(comparison_images[digit, 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"Digit {digit}", fontsize=11)
        ax.set(xticks=[], yticks=[])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if digit == 0:
            ax.set_ylabel("Input", fontsize=11)
    fig.add_subplot(grid[0, 10]).axis("off")

    for row, channel in enumerate(selected_channels, start=1):
        channel_maps = second_layer_maps[:, channel].cpu()
        channel_max = float(channel_maps.max())
        # A wholly zero row is labelled explicitly; use [0, 1] only as a display fallback.
        display_max = channel_max if channel_max > 0 else 1.0
        for digit in range(10):
            ax = fig.add_subplot(grid[row, digit])
            im = ax.imshow(channel_maps[digit], cmap="magma", vmin=0, vmax=display_max,
                           interpolation="nearest")
            ax.set(xticks=[], yticks=[])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if digit == 0:
                ax.set_ylabel(f"Ch {channel}", fontsize=11)
        color_axis = fig.add_subplot(grid[row, 10])
        if channel_max > 0:
            bar = fig.colorbar(im, cax=color_axis, ticks=[0, channel_max])
            bar.ax.set_yticklabels(["0", f"{channel_max:.2g}"])
            bar.ax.tick_params(labelsize=9)
        else:
            color_axis.axis("off")
            color_axis.text(.5, .5, "All\nzero", ha="center", va="center", fontsize=9)
    plt.show()


### From logits to predictions
For this ten-class problem, the model returns `[B, 10]` raw logits.
`CrossEntropyLoss(logits, labels)` takes these logits and integer labels of shape `[B]`.
Do not apply softmax before that loss.

For display, `softmax(logits, dim=1)` converts scores to probabilities summing to 1 per image.
`argmax(dim=1)` chooses the predicted class.
This differs from last week's single-logit binary example with `BCEWithLogitsLoss`.

The gallery below uses the 30 embedded examples for illustration. It is **not** a full test-set
accuracy estimate. The checkpoint's full test metrics are shown separately.

In [ ]:
if checkpoint is not None:
    with torch.no_grad():
        probabilities = cnn(demo_images).softmax(dim=1)
        confidence, predictions = probabilities.max(dim=1)
    image_grid(demo_images[:10], [
        f"True {int(y)} | pred {int(p)}\npredicted-class prob. {float(c):.2f}"
        for y, p, c in zip(demo_labels[:10], predictions[:10], confidence[:10])
    ])
    print("Correct among the 30 selected examples:", int((predictions == demo_labels).sum()), "/ 30")
    print("These selected examples do not estimate full-test accuracy.")
else:
    print("Pretrained prediction gallery will appear after loading tutorial6_cnn.pt.")

### Actual training curves and held-out errors
Training changes weights; evaluation only uses them.
The appendix records training and validation metrics after each epoch, then evaluates the
fixed final model on the official test set.

**Ask:** did learning occur? Does the validation trend follow training?
Higher training performance alone does not establish generalization.
The error gallery is deliberately selected to show mistakes, so its error rate is not representative.

In [ ]:
if checkpoint is not None:
    history = checkpoint.get("history", [])
    if history:
        epochs = [row["epoch"] for row in history]
        fig, axes = plt.subplots(1, 2, figsize=(9, 3))
        for split, label in [("train", "Train"), ("val", "Validation")]:
            axes[0].plot(epochs, [row[f"{split}_loss"] for row in history], marker="o", label=label)
            axes[1].plot(epochs, [row[f"{split}_accuracy"] for row in history], marker="o", label=label)
        axes[0].set(xlabel="Epoch", ylabel="Mean cross-entropy")
        axes[1].set(xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
        for ax in axes:
            ax.set_xticks(epochs)
            ax.legend()
        fig.tight_layout()
        plt.show()
    else:
        print("This checkpoint has no recorded learning curves.")
    print("Official held-out test metrics:", checkpoint.get("test_metrics", "Not stored"))
    gallery = checkpoint.get("error_gallery")
    if gallery is not None and len(gallery["labels"]) > 0:
        image_grid(gallery["images"], [
            f"True {int(y)} | pred {int(p)}"
            for y, p in zip(gallery["labels"], gallery["predictions"])
        ], columns=4)
    else:
        print("No error examples stored.")
else:
    print("No training results yet. The TA can generate them using the appendix.")

### The same building blocks form a 1D CNN

Return to Part A's signal. There we supplied a fixed difference kernel.
This network instead contains **four trainable 1D filters**, followed by ReLU, pooling and a classifier.
A training dataset and classification loss would be needed to learn useful filters and predictions.

Read the shapes to see the correspondence with the 2D CNN.
This example is **untrained**; its final scores do not demonstrate successful spike detection.

In [ ]:
torch.manual_seed(42)
signal_cnn = nn.Sequential(
    nn.Conv1d(1, 4, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool1d(2),
    nn.Flatten(start_dim=1),
    nn.Linear(4 * 50, 2),  # Example task: two class scores for a length-100 signal
)
with torch.no_grad():
    signal_values = signal_batch
    print("Input".ljust(18), tuple(signal_values.shape))
    for layer in signal_cnn:
        signal_values = layer(signal_values)
        print(type(layer).__name__.ljust(18), tuple(signal_values.shape))
print("Conv1d weights:", tuple(signal_cnn[0].weight.shape))
print("Conv1d parameters:", parameter_count(signal_cnn[0]))
print("This network has not been trained; its final scores are not meaningful predictions.")

## 9 · Exit questions

1. An input `[4, 3, 32, 32]` goes through `Conv2d(3, 8, 3, padding=1)`,
   then `ReLU`, then `MaxPool2d(2)`. What is its final shape? How many learned parameters?
2. What changes if we double the image width and height? What stays fixed?
3. Three sensors each record 100 time steps. How would you arrange one batched sample
   for a 1D convolution? Why is this not automatically a 3D convolution?

<details><summary>Reveal answers after discussion</summary>

1. `[4, 8, 16, 16]`; 224 parameters. ReLU and pooling have no learned parameters.
2. The final spatial size becomes `32×32`; the convolution still has 224 parameters.
   A later fixed-size linear classifier would need its input dimensions reconsidered.
3. `[1, 3, 100]`. The kernel slides along one sequence axis; the sensors are input channels.

</details>

**Take away:** CNNs learn local detectors and reuse them across positions.
Track spatial dimensions, channel dimensions and parameter counts separately.
The same training loop from last week can learn convolution weights.

**End of the 45-minute core lesson.**

## Optional extension · Same receptive field: compare parameters and computation (3–5 minutes)

Should we use **one `5×5` convolution** or **two `3×3` convolutions with a ReLU between them**?
The comparison asks how much each design costs for the same theoretical `5×5` receptive field
at an interior output. It does not assume that they compute the same function or have equal accuracy.

### Fix the comparison conditions first
- Input, hidden and output channel counts are all **C**; these are ordinary convolutions (`groups=1`).
- Stride and dilation are both 1.
- Padding preserves **H×W** at every layer: padding 2 for the `5×5`, padding 1 for each `3×3`.
- Count work for **one image**. Batch size B multiplies computation by B, but does not change parameters.

| Quantity | One 5×5 layer | Two 3×3 layers |
|---|---:|---:|
| Theoretical receptive field | 5×5 | 5×5 |
| Weights | $25C^2$ | $2(9C^2)=18C^2$ |
| Biases | $C$ | $2C$ |
| Total learned parameters | $25C^2+C$ | $18C^2+2C$ |
| Convolution multiply-accumulates (MACs) per image | $25HWC^2$ | $18HWC^2$ |

### Why are there fewer convolution calculations too?
For **one output value**, a convolution with spatial kernel size K combines $K^2C$ products.
There are $HWC$ output values, so a layer costs $HWK^2C^2$ MACs under these assumptions.
One MAC means one multiplication accumulated into a sum. This is a standard dense-convolution count;
it excludes bias additions, ReLU, memory transfers and implementation-specific optimizations.

The two smaller layers together cost $2\times9HWC^2=18HWC^2$, versus $25HWC^2$ for the larger layer.
Thus their convolution MAC count is **18/25 = 72%**, or **28% lower**.
The parameter reduction including biases is slightly different because the two-layer design has two bias vectors.

**Predict before running:** for C=16 and H=W=28, which counts change if the image size doubles?
Which counts change if the batch size doubles?
The next cell computes the costs from the actual layer configurations and output shapes.


In [ ]:
C, H, W = 16, 28, 28
wide_layer = nn.Conv2d(C, C, 5, padding=2)
stacked_layers = nn.Sequential(
    nn.Conv2d(C, C, 3, padding=1), nn.ReLU(),
    nn.Conv2d(C, C, 3, padding=1),
)
cost_input = torch.zeros(1, C, H, W)
with torch.no_grad():
    wide_output = wide_layer(cost_input)
    first_output = stacked_layers[0](cost_input)
    stacked_output = stacked_layers[2](stacked_layers[1](first_output))
assert wide_output.shape == first_output.shape == stacked_output.shape == cost_input.shape

def convolution_macs_per_image(layer, output):
    out_h, out_w = output.shape[-2:]
    kernel_h, kernel_w = layer.kernel_size
    products_per_output = (layer.in_channels // layer.groups) * kernel_h * kernel_w
    return out_h * out_w * layer.out_channels * products_per_output

wide_macs = convolution_macs_per_image(wide_layer, wide_output)
first_macs = convolution_macs_per_image(stacked_layers[0], first_output)
second_macs = convolution_macs_per_image(stacked_layers[2], stacked_output)
stacked_macs = first_macs + second_macs
assert wide_macs == 25 * H * W * C**2
assert stacked_macs == 18 * H * W * C**2

print(f"One image: input and every layer output have shape {tuple(cost_input.shape)}")
print("Stride = dilation = 1; channels stay at C = 16; spatial size stays at 28 x 28.")
print(f"{'Design':<18} {'Parameters (with bias)':>24} {'Convolution MACs/image':>24}")
print(f"{'One 5 x 5':<18} {parameter_count(wide_layer):>24,} {wide_macs:>24,}")
print(f"{'Two 3 x 3':<18} {parameter_count(stacked_layers):>24,} {stacked_macs:>24,}")
print(f"Two-layer MACs: {first_macs:,} + {second_macs:,} = {stacked_macs:,}")
print(f"Convolution MAC reduction: {(1 - stacked_macs / wide_macs):.0%}")
print("This is an operation-count comparison, not a wall-clock speed benchmark.")


### Interpret the result carefully

- **Under these conditions**, two `3×3` layers use fewer parameters and fewer convolution MACs
  for the same theoretical receptive field. This is one reason to consider smaller stacked kernels.
- **Fewer MACs do not guarantee lower latency.** Two layers introduce another layer invocation and
  an intermediate feature map, plus the ReLU operation. Memory traffic, hardware and kernel implementations
  also affect runtime. The cell above does not measure runtime.
- **Same receptive field does not mean the same function.** The extra ReLU changes what can be represented;
  this comparison alone says nothing about which model will be more accurate.
- **The assumptions matter.** Changing the hidden channel count, stride or padding changes the calculation.
  Without size-preserving padding, the two `3×3` layers have different spatial output sizes, so the
  $18HWC^2$ expression above no longer applies directly.

For the prediction questions: doubling both H and W gives four times the convolution MACs and
unchanged parameter counts. Doubling batch size doubles MACs and also leaves parameters unchanged.

## After-class / TA preparation · Train and export the classroom checkpoint

**Skip during class.** This section is the reproducible preparation path for the model in Section 8.

1. Run Setup and define `TinyCNN` and the checkpoint constants in Section 8.
2. Run the training helper cell below.
3. Set `RUN_TRAINING = True` in the next cell; adjust the configuration, then run it.
4. The file `tutorial6_cnn.pt` will contain weights, curves, test metrics and up to eight errors.
5. Set `RUN_TRAINING = False` again before distributing the notebook.
6. Rerun the checkpoint-loading cell and Section 8 to display the actual results; save the notebook.
   Distribute the checkpoint alongside it. In Colab, download the checkpoint before ending the session.

Training downloads MNIST through `torchvision`. It uses the same preprocessing as the embedded
examples: grayscale pixels scaled to `[0,1]`, no additional normalization.
The default uses 12,000 training images and 2,000 separate validation images from the official
training split; the remaining training images are unused. Increase these counts if desired.
The official 10,000-image test split is evaluated only after the configured training run.

Choose the settings before examining test results. Use validation, not repeated test evaluation,
to make development decisions. The exported model is the **final epoch**, not a test-selected model.
CPU is supported; CUDA or MPS can optionally speed up preparation.

In [ ]:
# The helper defines training but does not start it.
def evaluate_model(model, loader, device):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += F.cross_entropy(logits, labels, reduction="sum").item()
            correct += (logits.argmax(1) == labels).sum().item()
            count += len(labels)
    return {"loss": total_loss / count, "accuracy": correct / count, "count": count}

def train_epoch(model, loader, optimizer, device):
    model.train()
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()

def collect_errors(model, loader, device, maximum=8):
    images_out, labels_out, predictions_out = [], [], []
    model.eval()
    with torch.no_grad():
        for images, labels in loader:
            predictions = model(images.to(device)).argmax(1).cpu()
            for i in torch.where(predictions != labels)[0].tolist():
                images_out.append(images[i].cpu())
                labels_out.append(int(labels[i]))
                predictions_out.append(int(predictions[i]))
                if len(images_out) == maximum:
                    break
            if len(images_out) == maximum:
                break
    return {
        "images": torch.stack(images_out) if images_out else torch.empty(0, 1, 28, 28),
        "labels": torch.tensor(labels_out, dtype=torch.long),
        "predictions": torch.tensor(predictions_out, dtype=torch.long),
    }

def prepare_checkpoint(config, destination):
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, Subset
    import time

    if destination.exists():
        raise FileExistsError(f"{destination} exists. Choose a new path to preserve the existing result.")
    torch.manual_seed(config["seed"])
    device = torch.device(config["device"])
    official_train = datasets.MNIST("./data", train=True, download=True, transform=transforms.ToTensor())
    official_test = datasets.MNIST("./data", train=False, download=True, transform=transforms.ToTensor())
    n_train, n_val = config["train_size"], config["val_size"]
    if min(n_train, n_val, config["epochs"], config["batch_size"]) < 1 or n_train + n_val > len(official_train):
        raise ValueError("Use positive sizes/epochs and train_size + val_size <= 60000.")
    split_generator = torch.Generator().manual_seed(config["seed"])
    order = torch.randperm(len(official_train), generator=split_generator)
    train_indices, val_indices = order[:n_train], order[n_train:n_train + n_val]
    train_subset = Subset(official_train, train_indices.tolist())
    val_subset = Subset(official_train, val_indices.tolist())
    shuffle_generator = torch.Generator().manual_seed(config["seed"])
    train_loader = DataLoader(train_subset, batch_size=config["batch_size"], shuffle=True,
                              generator=shuffle_generator, num_workers=0)
    train_eval_loader = DataLoader(train_subset, batch_size=256, shuffle=False, num_workers=0)
    val_loader = DataLoader(val_subset, batch_size=256, shuffle=False, num_workers=0)
    test_loader = DataLoader(official_test, batch_size=256, shuffle=False, num_workers=0)

    model = TinyCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    history = []
    start = time.perf_counter()
    for epoch in range(1, config["epochs"] + 1):
        train_epoch(model, train_loader, optimizer, device)
        # Both curves evaluate the weights at the END of the epoch.
        train_metrics = evaluate_model(model, train_eval_loader, device)
        val_metrics = evaluate_model(model, val_loader, device)
        row = {"epoch": epoch, "train_loss": train_metrics["loss"],
               "train_accuracy": train_metrics["accuracy"], "val_loss": val_metrics["loss"],
               "val_accuracy": val_metrics["accuracy"]}
        history.append(row)
        print(f"Epoch {epoch}: train loss={row['train_loss']:.4f}, acc={row['train_accuracy']:.3f}; "
              f"val loss={row['val_loss']:.4f}, acc={row['val_accuracy']:.3f}", flush=True)

    test_metrics = evaluate_model(model, test_loader, device)
    errors = collect_errors(model, test_loader, device)
    bundle = {
        "architecture": ARCHITECTURE_ID, "preprocessing": PREPROCESSING_ID,
        "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "config": dict(config), "history": history, "test_metrics": test_metrics,
        "error_gallery": errors, "train_indices": train_indices, "val_indices": val_indices,
        "elapsed_seconds": time.perf_counter() - start,
        "torch_version": str(torch.__version__),
        "selection": "final_epoch",
    }
    torch.save(bundle, destination)
    # Confirm that classroom loading reconstructs the same CPU predictions.
    restored, _ = load_teaching_checkpoint(destination)
    model = model.cpu().eval()
    with torch.no_grad():
        torch.testing.assert_close(restored(demo_images), model(demo_images))
    print("Test metrics:", test_metrics)
    print("Saved and reload-checked:", destination.resolve())
    return bundle

In [ ]:
RUN_TRAINING = True  # Keep False for class and for the distributed notebook.
TRAIN_CONFIG = {
    "seed": 42,
    "train_size": 12000,
    "val_size": 2000,
    "epochs": 3,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "device": "cpu",  # Optionally "cuda" or "mps" if available.
}
TRAIN_OUTPUT_PATH = Path("tutorial6_cnn.pt")

if RUN_TRAINING:
    prepared_bundle = prepare_checkpoint(TRAIN_CONFIG, TRAIN_OUTPUT_PATH)
else:
    print("Training is disabled. Class demonstrations do not wait for training.")

### Reproduction notes
- `model.eval()` sets evaluation behavior; `torch.no_grad()` disables gradient recording.
  This particular network has no dropout or batch normalization, but the two operations still
  have different roles.
- Loss is accumulated by sample count in evaluation, so a smaller final batch is weighted correctly.
- Random seeds and saved split indices help reproduction. Exact numerical results may differ
  across devices and library versions.
- The checkpoint is for inference and teaching. It does not include optimizer state for exact
  resumption of training.